# Corpus Frequency — Simplex vs. Root+*hata* Minimal Pairs

Log-transformed frequency from two independently tokenized corpora: KoFREN (Bareun tokenizer) and NSMC (mecab-ko).

In [1]:
import math
import pandas as pd

PAIRS = [
    ("헤아리다", ("헤아리", "VV"), "생각하다", ("생각하", "VV")),
    ("부르다", ("부르", "VV"), "노래하다", ("노래하", "VV")),
    ("게우다", ("게우", "VV"), "토하다", ("토하", "VV")),
    ("굽다", ("굽", "VV"), "요리하다", ("요리하", "VV")),
    ("덥다", ("덥", "VA"), "따뜻하다", ("따뜻하", "VA")),
]

def lookup(df, stem, pos):
    sub = df[(df["WORD"] == stem) & (df["POS_TAG"] == pos)]
    return int(sub["COUNT"].sum()) if not sub.empty else 0

In [2]:
kofren = pd.read_csv("kofren_all_speakers.csv")

rows = []
for simplex, (s_stem, s_pos), hada, (h_stem, h_pos) in PAIRS:
    s_count = lookup(kofren, s_stem, s_pos)
    h_count = lookup(kofren, h_stem, h_pos)
    s_log = math.log10(s_count) if s_count > 0 else 0.0
    h_log = math.log10(h_count) if h_count > 0 else 0.0
    rows.append(["simplex", simplex, s_stem, s_pos, s_count, s_log])
    rows.append(["hada", hada, h_stem, h_pos, h_count, h_log])

kofren_df = pd.DataFrame(rows, columns=["role", "word", "stem", "pos", "count", "log10_count"])
kofren_df.to_csv("kofren_frequency_results.csv", index=False, encoding="utf-8-sig")
kofren_df

,role,word,stem,pos,count,log10_count
0,simplex,헤아리다,헤아리,VV,164,2.214844
1,hada,생각하다,생각하,VV,77129,4.887218
2,simplex,부르다,부르,VV,26646,4.425632
3,hada,노래하다,노래하,VV,1783,3.251151
4,simplex,게우다,게우,VV,0,0.000000
5,hada,토하다,토하,VV,1934,3.286456
6,simplex,굽다,굽,VV,8159,3.911637
7,hada,요리하다,요리하,VV,2955,3.470557
8,simplex,덥다,덥,VA,10644,4.027105
9,hada,따뜻하다,따뜻하,VA,13157,4.119157


## Cross-Register Replication (NSMC)

NSMC counts were obtained separately (mecab-ko, Google Cloud Shell) and are loaded here from the saved results file for comparison.

In [3]:
nsmc_df = pd.read_csv("nsmc_frequency_results.csv")
nsmc_df

,role,word,count,log10_count
0,simplex,헤아리다,3,0.477121
1,hada,생각하다,1464,3.165570
2,simplex,부르다,334,2.523747
3,hada,노래하다,32,1.505150
4,simplex,게우다,1,0.000000
5,hada,토하다,22,1.342423
6,simplex,굽다,10,1.000000
7,hada,요리하다,5,0.698970
8,simplex,덥다,26,1.414973
9,hada,따뜻하다,217,2.336460


In [4]:
concept_map = {("헤아리다", "생각하다"): "think", ("부르다", "노래하다"): "sing",
               ("게우다", "토하다"): "vomit", ("굽다", "요리하다"): "cook",
               ("덥다", "따뜻하다"): "warm"}

kofren_wide = kofren_df.pivot_table(index="word", columns="role", values="log10_count", aggfunc="first")
nsmc_wide = nsmc_df.pivot_table(index="word", columns="role", values="log10_count", aggfunc="first")

summary = []
for (simplex, hada), concept in concept_map.items():
    row = {"concept": concept, "simplex": simplex, "hada": hada}
    for corpus, wide in [("kofren", kofren_wide), ("nsmc", nsmc_wide)]:
        s, h = wide.loc[simplex, "simplex"], wide.loc[hada, "hada"]
        row[f"{corpus}_direction"] = "simplex > hada (matches hypothesis)" if s > h else "hada > simplex (opposite)"
    summary.append(row)

pd.DataFrame(summary).set_index("concept")

,simplex,hada,kofren_direction,nsmc_direction
concept,,,,
think,헤아리다,생각하다,hada > simplex (opposite),hada > simplex (opposite)
sing,부르다,노래하다,simplex > hada (matches hypothesis),simplex > hada (matches hypothesis)
vomit,게우다,토하다,hada > simplex (opposite),hada > simplex (opposite)
cook,굽다,요리하다,simplex > hada (matches hypothesis),simplex > hada (matches hypothesis)
warm,덥다,따뜻하다,hada > simplex (opposite),hada > simplex (opposite)
